In [ ]:
import geopandas as gpd
from debugpy.common.log import log_dir
from oggm import cfg, utils, workflow, tasks, graphics
from oggm.core import gis
import matplotlib.pyplot as plt
from oggm.workflow import execute_entity_task
from oggm.utils import get_demo_file
cfg.initialize(logging_level='WARNING')
cfg.initialize()
import geopandas as gpd
import pandas as pd
from shapely.ops import unary_union
cfg.PARAMS['border'] = 360

# Read the files
gdf_cgi_12 = gpd.read_file(r"D:\oggm_related\oggm_tutorials\OGGM_Qilian\qilian_1850_big\outline_base_firstGlacier\qilian_big_only_outline\qilian_big_baseFirst_outline.shp")
rgi13 = gpd.read_file(r"D:\oggm_related\Glacier_little\rgi\RGIV62\13_rgi62_CentralAsia\13_rgi62_CentralAsia.shp")
gdf_cgi_489 = gpd.read_file(r"D:\oggm_related\Glacier_little\Myproject5\Qilianshan_Outline_LIA_489\Qilianshan_Outline_LIA_489.shp")

# Cook the RGI data
cfg.PARAMS['store_model_geometry'] = True
gdf_cgi_12_cook = utils.cook_rgidf(gdf_cgi_12, o1_region='13', assign_column_values={'RGIID':'RGIId'})
gdf_cgi_489_cook = utils.cook_rgidf(gdf_cgi_489, o1_region='13', assign_column_values={'RGIID':'RGIId'})

# Get the list of RGIIds for the 12 large glaciers
cgi_rgis = gdf_cgi_12_cook['RGIId'].tolist()
# Select the 12 large glaciers from RGI13
rgi13_qilian_Big = rgi13[rgi13['RGIId'].isin(cgi_rgis)]

In [ ]:
import sys
print(sys.executable)

In [ ]:
# 查看安装位置和内容
import importlib.util
spec = importlib.util.find_spec("agile")
print(f"agile 位置: {spec.origin if spec else '未找到'}")

In [ ]:
import agile1d
import agile2d
import os

# agile1d 子模块
agile1d_path = agile1d.__path__[0]
print("agile1d 内容:")
for f in os.listdir(agile1d_path):
    print(f"  {f}")

# agile2d 子模块
agile2d_path = agile2d.__path__[0]
print("\nagile2d 内容:")
for f in os.listdir(agile2d_path):
    print(f"  {f}")

In [ ]:
import geopandas as gpd
import pandas as pd
lia = gpd.read_file("/mnt/e/2Data/022_工作2/出图数据/Himalaya_LIA_with_RGIID_Complete/Himalaya_LIA_with_RGIID_Complete.shp")
rgi = gpd.read_file("/mnt/e/2Data/022_工作2/出图数据/himalaya_glaciers_complete/himalaya_glaciers_complete.shp")
# 筛选 RGI 面积 >= 10 km²
rgi_large = rgi[rgi['Area'] >= 10].copy()
large_rgi_ids = set(rgi_large['RGIId'])

# 找包含这些大冰川的 LIA
def contains_large_rgi(rgi_str):
    if pd.isna(rgi_str):
        return False
    ids = [x.strip() for x in str(rgi_str).split(';')]
    return bool(set(ids) & large_rgi_ids)

lia_matched = lia[lia['AllRGIs'].apply(contains_large_rgi)].copy()

print("LIA筛选完成: {} 条".format(len(lia_matched)))
# 筛选MainRGI对应的RGI冰川
main_rgi_ids = lia_matched['MainRGI'].tolist()
rgi_to_calibrate = rgi[rgi['RGIId'].isin(main_rgi_ids)].copy()

# 转WGS84
lia_matched = lia_matched.to_crs("EPSG:4326")
rgi_to_calibrate = rgi_to_calibrate.to_crs("EPSG:4326")

# 建立映射字典: LIA的UniqueID → MainRGI
lia_to_main_rgi = dict(zip(
    lia_matched['UniqueID'].astype(int),
    lia_matched['MainRGI']
))

print("RGI待校准: {} 条".format(len(rgi_to_calibrate)))
print("映射关系: {} 条".format(len(lia_to_main_rgi)))
from oggm import cfg, workflow, tasks, utils

WORKING_DIR = "/mnt/f/oggm_related/oggm_rgi_calibration"
DEM_PATH = "/mnt/d/01 文件整理/04 数据/HMA_newregion/DEM_clip/Himalaya_3regions_DEM.tif"
MB_PATHS = [
    "/mnt/d/01 文件整理/04 数据/03 物质平衡数据/hugonnet time_series_14/dh_14_rgi60_pergla_rates.csv",
    "/mnt/d/01 文件整理/04 数据/03 物质平衡数据/hugonnet time_series_15/dh_15_rgi60_pergla_rates.csv",
]
MB_PERIOD = "2000-01-01_2020-01-01"
# 读取MB数据
dfs = []
for path in MB_PATHS:
    print("读取: {}".format(path))
    df = pd.read_csv(path)
    dfs.append(df)
mb_all = pd.concat(dfs, ignore_index=True)
mb_data = mb_all[mb_all['period'] == MB_PERIOD].copy()
mb_data['dmdtda_kg'] = mb_data['dmdtda'] * 1000
mb_data = mb_data.set_index('rgiid')
print("MB数据: {} 条".format(len(mb_data)))
# 清理RGI数据
rgi = rgi_to_calibrate.copy()
rgi = rgi[rgi.geometry.notna() & ~rgi.is_empty].copy()
rgi['geometry'] = rgi.geometry.buffer(0)
rgi = rgi[rgi.is_valid].copy()
rgi = rgi[rgi['RGIId'].isin(mb_data.index)].copy()
print("有MB数据的RGI冰川: {}".format(len(rgi)))
from oggm import cfg, workflow, tasks, GlacierDirectory
cfg.initialize()

cfg.initialize(logging_level='WARNING')
cfg.PATHS['working_dir'] = WORKING_DIR
cfg.PATHS['dem_file'] = DEM_PATH
cfg.PARAMS['border'] = 360
cfg.PARAMS['use_intersects'] = False
cfg.PARAMS['use_rgi_area'] = False
cfg.PARAMS['continue_on_error'] = True
cfg.PARAMS['use_multiprocessing'] = True
cfg.PARAMS['mp_processes'] = 20

gdirs_rgi = workflow.init_glacier_directories(rgi, reset=True)
print("初始化完成: {}".format(len(gdirs_rgi)))
for gdir in gdirs_rgi:
    tasks.define_glacier_region(gdir, source='USER')
print("define_glacier_region完成: {}".format(len(gdirs_rgi)))
from oggm.workflow import execute_entity_task
cfg.PARAMS['use_multiprocessing'] = True

execute_entity_task(tasks.glacier_masks, gdirs_rgi)
execute_entity_task(tasks.compute_centerlines, gdirs_rgi)
execute_entity_task(tasks.initialize_flowlines, gdirs_rgi)
execute_entity_task(tasks.compute_downstream_line, gdirs_rgi)
execute_entity_task(tasks.compute_downstream_bedshape, gdirs_rgi)
execute_entity_task(tasks.catchment_area, gdirs_rgi)
execute_entity_task(tasks.catchment_intersections, gdirs_rgi)
execute_entity_task(tasks.catchment_width_geom, gdirs_rgi)
execute_entity_task(tasks.catchment_width_correction, gdirs_rgi)
print("预处理完成")
cfg.PARAMS['baseline_climate'] = 'W5E5'
for gdir in gdirs_rgi:
    tasks.process_w5e5_data(gdir)#不需要了
# execute_entity_task(tasks.process_cru_data, gdirs_rgi)
print("气候数据处理完成")
from oggm.core.massbalance import mb_calibration_from_scalar_mb
# cfg.PARAMS['use_winter_prcp_factor'] = False
cfg.PARAMS['prcp_fac'] = None

from oggm.core.massbalance import mb_calibration_from_scalar_mb

cfg.PARAMS['use_winter_prcp_factor'] = False
cfg.PARAMS['prcp_fac'] = None

success = 0
fail = 0
failed_list = []

for gdir in gdirs_rgi:
    try:
        rgi_id = gdir.rgi_id
        ref_mb = float(mb_data.loc[rgi_id, 'dmdtda_kg'])
        mb_calibration_from_scalar_mb(
            gdir,
            ref_mb=ref_mb,
            overwrite_gdir=True,
            ref_mb_years=(2000, 2020),
            calibrate_param1='prcp_fac',   # 第一步：降水因子
            calibrate_param2='melt_f',     # 第二步：融化因子
            calibrate_param3='temp_bias',  # 第三步：温度偏差
        )
        success += 1
    except Exception as e:
        fail += 1
        failed_list.append((gdir.rgi_id, str(e)[:80]))
        print(f"失败: {gdir.rgi_id} - {str(e)[:60]}")

print(f"校准完成: 成功 {success}, 失败 {fail}")